### Import Dependencies

In [24]:
import os
from google import genai
import instructor
from pydantic import BaseModel, Field
import pandas as pd
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

COLLECTION_NAME = "Amazon-items-minilm-l6-v2"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")

/Users/ganesh.m/Projects/ai-engineering-bootcamp/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Create Qdrant colection for hybrid search

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [45]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
        "sentence-transformers-all-MiniLM-L6-v2": VectorParams(size=384, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

In [5]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [ ]:
# def get_embedding(text, model="text-embedding-3-small"):    
#     response = openai.embeddings.create(
#         input=text,
#         model=model,
#     )
#     return response.data[0].embedding

In [6]:
def get_embedding(text, model="sentence-transformers/all-MiniLM-L6-v2"):
    embedding_model = SentenceTransformer(model)
    return embedding_model.encode(text, normalize_embeddings=True).tolist()

In [34]:
def get_embeddings_batch(text_list, model="sentence-transformers/all-MiniLM-L6-v2", batch_size=100):
    embedding_model = SentenceTransformer(model)
    if len(text_list) <= batch_size:
        return embedding_model.encode(text_list, normalize_embeddings=True).tolist()
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response =  embedding_model.encode(batch, normalize_embeddings=True).tolist()
        # response = openai.embeddings.create(input=batch, model=model)
        print(response)
        all_embeddings.extend(response)
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [8]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [9]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


### Preprocess title and features

In [10]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [11]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [12]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [13]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...","Bosttor Bluetooth Beanie Hat with Light, Headl...",26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51b7qcj8dZ...
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41WsGRr-3T...
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41fsjaknf1...
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51YFypeuZh...
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41V7Wi2ckb...


In [14]:
list(df_items["description"].items())[0]

(0,
 'Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It point

In [15]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_.jpg')

In [16]:
data_to_embed = df_items[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [17]:
data_to_embed

[{'description': 'Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet aw

In [18]:
len(data_to_embed)

1000

In [19]:
text_to_embed = [item["description"] for item in data_to_embed]

In [20]:
text_to_embed

['Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It points di

In [36]:
embeddings = get_embeddings_batch(text_to_embed)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6952.82it/s]


[[-0.05383753031492233, 0.02415633760392666, 0.002811064012348652, -0.009138744324445724, 0.06782281398773193, 0.02624356560409069, 0.06409450620412827, 0.06742851436138153, -0.04205457866191864, 0.007416084408760071, 0.07410325855016708, 0.017220942303538322, 0.03724734112620354, -0.062480948865413666, 0.0360555499792099, 0.04776619374752045, 0.07822027802467346, -0.06578341871500015, 0.017514539882540703, -0.03819265216588974, 0.024362459778785706, -0.01278739981353283, 0.0744662806391716, 0.07515056431293488, -0.02213413640856743, 0.007027321495115757, 0.04860519617795944, -0.02384836971759796, -0.06626036018133163, -0.019885098561644554, 0.009347768500447273, 0.01734270714223385, 0.043090078979730606, -0.04283245652914047, -0.07025843858718872, 0.01026200968772173, 0.09850554168224335, 0.023098427802324295, -0.027153857052326202, 0.035200051963329315, -0.03896116837859154, -0.030760422348976135, 0.025853795930743217, -0.026537593454122543, 0.032849494367837906, 0.005637515336275101

In [37]:
len(embeddings)

1000

In [42]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "sentence-transformers-all-MiniLM-L6-v2": embedding,
                "bm25": Document(
                    text=data["description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i += 1

In [43]:
pointstructs[0].vector

{'sentence-transformers-all-MiniLM-L6-v2': [-0.05383753031492233,
  0.02415633760392666,
  0.002811064012348652,
  -0.009138744324445724,
  0.06782281398773193,
  0.02624356560409069,
  0.06409450620412827,
  0.06742851436138153,
  -0.04205457866191864,
  0.007416084408760071,
  0.07410325855016708,
  0.017220942303538322,
  0.03724734112620354,
  -0.062480948865413666,
  0.0360555499792099,
  0.04776619374752045,
  0.07822027802467346,
  -0.06578341871500015,
  0.017514539882540703,
  -0.03819265216588974,
  0.024362459778785706,
  -0.01278739981353283,
  0.0744662806391716,
  0.07515056431293488,
  -0.02213413640856743,
  0.007027321495115757,
  0.04860519617795944,
  -0.02384836971759796,
  -0.06626036018133163,
  -0.019885098561644554,
  0.009347768500447273,
  0.01734270714223385,
  0.043090078979730606,
  -0.04283245652914047,
  -0.07025843858718872,
  0.01026200968772173,
  0.09850554168224335,
  0.023098427802324295,
  -0.027153857052326202,
  0.035200051963329315,
  -0.0389611

In [46]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [47]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [51]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="sentence-transformers-all-MiniLM-L6-v2",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [52]:
results = retrieve_data("Can I get a tablet?", qdrant_client, k=20)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7692.96it/s]


In [53]:
results

{'retrieved_context_ids': ['B09RHF4L45',
  'B0BGHBL86V',
  'B0BN58Z4YX',
  'B0C5QQDN3T',
  'B0B157WDDJ',
  'B0BHVH4D37',
  'B0B6ZZH83Y',
  'B0B58CPFWY',
  'B0B159KDFP',
  'B09WR36NK8',
  'B0BSD3QK7M',
  'B0BJ2YWMTP',
  'B0BCFYCXRH',
  'B0C7DCS2KW',
  'B0BWRZ1HRD',
  'B09W2BMG5N',
  'B09XQMRYBJ',
  'B09PYV6B4B',
  'B09SZNLXJQ',
  'B0B3QVP3W3'],
 'retrieved_context': ['plimpton Android Tablet 10 inch - 8 Core Processor, 4GB RAM 64GB Storage, 1.8M Length USB-C Cable, Android 11, 1920 x 1200 FHD IPS, 13MP Sony Rear Camera, 6000mAh Battery, 2.4G+5G WiFi Tablets PC High Performance Tablet 10.1 inch: Features a faster 1.8GHz Octa-Core CPU and 4GB RAM, you can not only obtain seamless multitasking and hyper-fast content streaming experience, but also can view social media, watch videos, play games, complete homework or finish work. With a 64GB internal storage and support a SD card expanded to 512GB(NOT included), no longer worry about insufficient memory.★★Please note this is a Wi-Fi tablet, 

### Hybrid Search with weighted RRF

In [ ]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="sentence-transformers-all-MiniLM-L6-v2",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [ ]:
results = retrieve_data("Can I get a tablet?", qdrant_client, k=20)

In [ ]:
results